In [2]:
import numpy as np
import pandas as pd
df=pd.read_csv("../data/raw/PS_20174392719_1491204439457_log.csv")
print(df.shape)

(6362620, 11)


In [3]:
df["orig_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["dest_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

df["orig_amount_error"] = (
    df["oldbalanceOrg"]
    - df["amount"]
    - df["newbalanceOrig"]
).abs()

df["dest_amount_error"] = (
    df["oldbalanceDest"]
    + df["amount"]
    - df["newbalanceDest"]
).abs()

print(df.shape)
print(df.head())

(6362620, 15)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  \
0  M1979787155             0.0             0.0        0               0   
1  M2044282225             0.0             0.0        0               0   
2   C553264065             0.0             0.0        1               0   
3    C38997010         21182.0             0.0        1               0   
4  M1230701703             0.0             0.0        0               0   

   orig_balance_change  dest_balance_change  orig_amount_error  \
0              983

In [4]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
# USE
# ├── step
# ├── type
# ├── amount
# ├── oldbalanceOrg
# ├── newbalanceOrig
# ├── oldbalanceDest
# ├── newbalanceDest
# ├── orig_balance_change
# ├── dest_balance_change
# ├── orig_amount_error
# └── dest_amount_error

# TARGET
# └── isFraud

# REMOVE FOR NOW
# ├── nameOrig
# ├── nameDest
# └── isFlaggedFraud


feature_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_amount_error",
    "dest_amount_error"
]

X = df[feature_columns]
y = df["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

X shape: (6362620, 11)
y shape: (6362620,)
Training: (5090096, 11)
Testing: (1272524, 11)


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["type"]

numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_amount_error",
    "dest_amount_error"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

In [7]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)

Train: (5090096, 15)
Test: (1272524, 15)


In [8]:
preprocessor.fit_transform(X_train)
preprocessor.transform(X_test)

array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         8.9057721e+05,  8.9035921e+05,  0.0000000e+00],
       [ 1.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
        -9.7734240e+04,  1.9546848e+05,  1.9546848e+05],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         0.0000000e+00,  5.5924100e+03,  5.9074100e+03],
       ...,
       [ 1.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
        -1.3566828e+05,  2.7133656e+05,  2.7133656e+05],
       [ 0.0000000e+00,  1.0000000e+00,  0.0000000e+00, ...,
         1.4438847e+05,  1.4438846e+05,  1.0000000e-02],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         2.4231185e+05,  2.3658285e+05,  0.0000000e+00]],
      shape=(1272524, 15))

In [9]:
print(type(X_train_processed))
print(X_train_processed.shape)


<class 'numpy.ndarray'>
(5090096, 15)


In [10]:
# Dummy Baseline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(X_train_processed, y_train)

dummy_pred = dummy.predict(X_test_processed)
print("Accuracy :", accuracy_score(y_test, dummy_pred))
print("Precision:", precision_score(y_test, dummy_pred, zero_division=0))
print("Recall   :", recall_score(y_test, dummy_pred, zero_division=0))
print("F1       :", f1_score(y_test, dummy_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dummy_pred))


Accuracy : 0.9987088652159016
Precision: 0.0
Recall   : 0.0
F1       : 0.0

Confusion Matrix:
[[1270881       0]
 [   1643       0]]


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Take 500,000 training rows
X_train_small = X_train_processed[:500000]
y_train_small = y_train.iloc[:500000]

# Train model
lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=50,
    solver="liblinear"
)

lr_model.fit(X_train_small, y_train_small)

# Predictions
lr_pred = lr_model.predict(X_test_processed)
lr_prob = lr_model.predict_proba(X_test_processed)[:, 1]

# Results
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred, zero_division=0))
print("Recall   :", recall_score(y_test, lr_pred, zero_division=0))
print("F1       :", f1_score(y_test, lr_pred, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, lr_prob))
print("PR-AUC   :", average_precision_score(y_test, lr_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))

Accuracy : 0.9592542066004256
Precision: 0.029341732755873033
Recall   : 0.9525258673158856
F1       : 0.05692979265187341
ROC-AUC  : 0.9912761173251667
PR-AUC   : 0.5422219625235902

Confusion Matrix:
[[1219109   51772]
 [     78    1565]]
